In [1]:
import os
import sys

# 1. sys.path에서 추가된 경로를 제거
if '/content/src' in sys.path:
    sys.path.remove('/content/src')

# 2. 심볼릭 링크 삭제
if os.path.islink('/content/src'):
    os.unlink('/content/src')

# # 3. Google Drive 언마운트
# drive.flush_and_unmount()  # Google Colab에서 사용하는 Google Drive 언마운트 메소드

In [2]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

my_path = '/content/src'
save_path = '/content/drive/MyDrive/Minimap Server/src'

if os.path.exists(my_path):
  os.remove(my_path)

os.symlink(save_path, my_path)
sys.path.insert(0, my_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content/drive/MyDrive/Minimap Server

/content/drive/MyDrive/Minimap Server


In [4]:
!pip install yt-dlp
!pip install ffmpeg
!pip install scipy
!pip install fastapi uvicorn python-multipart pyngrok

In [5]:
from line_extremities import process_video
from minimap_maker import create_smooth_video
import json

checkpoint = "./train_59.pt"
resolution_width = 455
resolution_height = 256
pp_radius = 4
pp_maxdists = 30
num_points_lines = 2

with open('coordinates.json', 'r') as f:
    coordinates_data = json.load(f)

In [6]:
import yt_dlp as youtube_dl
import io

def youtube_download(youtube_url):
    video_id_suffix = youtube_url[-4:]
    filename = f"video_{video_id_suffix}.mp4"

    ydl_opts = {
        'format': 'best',
        'outtmpl': filename,
        'noplaylist': True,
    }

    with youtube_dl.YoutubeDL(ydl_opts) as ydl:
        try:
            info_dict = ydl.extract_info(youtube_url, download=True)  # Download the video
            video_title = info_dict.get('title', None)
        except youtube_dl.utils.ExtractorError as e:
            print(f"Error extracting video information: {e}")
            return None, None

    # Read the downloaded video into memory
    video_bytes = io.BytesIO()
    with open(filename, 'rb') as f:
        video_bytes.write(f.read())
    video_bytes.seek(0)

    # Optionally, delete the file after reading into memory
    os.remove(filename)

    return video_bytes

In [7]:
labels_path = "./new_1"
field_path = "./field.jpg"
output_path = "./outputs/final_video_31.mp4"

In [10]:
import cv2

def process_video_with_status_update(task_id: str, video_bytes, checkpoint, resolution_width, resolution_height, pp_radius, pp_maxdists, num_points_lines, coordinates_data):
    result = process_video(video_bytes, checkpoint, resolution_width, resolution_height, pp_radius, pp_maxdists, num_points_lines, coordinates_data)
    # task_status[task_id] = {"status": "completed", "result": result}

    json_compatible_result = json.loads(json.dumps(result, cls=NumpyEncoder))
    # # 데이터 저장
    # with open('raw_data_31.json', 'w') as f:
    #     json.dump(json_compatible_result, f)
    smoothed_data = create_smooth_video(json_compatible_result, labels_path)
    # 데이터 저장
    with open('smoothed_data_31.json', 'w') as f:
        json.dump(smoothed_data, f)
    task_status[task_id] = {"status": "completed", "result": smoothed_data}


    field = cv2.imread(field_path)

    output_size = (1050, 680)
    fps = 30
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(output_path, fourcc, fps, output_size)

    for frame in smoothed_data:
        background = field.copy()
        players = smoothed_data[frame]
        for player in players:
            player_point = (int(player['x']), int(player['y']))
            radius = 10
            if player['team'] == 0:
                color = (0, 0, 255)  # Blue
            elif player['team'] == 1:
                color = (255, 0, 0)  # Red
            else:
                color = (255, 255, 255)  # White
            thickness = -1  # Fill the circle
            cv2.circle(background, player_point, radius, color, thickness)
        out.write(background)

    out.release()

In [ ]:
import asyncio
import nest_asyncio
import json
from fastapi import FastAPI, File, UploadFile, BackgroundTasks
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok
import numpy as np
from pydantic import BaseModel
from typing import Dict
from uuid import uuid4

nest_asyncio.apply()

app = FastAPI()

# In-memory store for task status (in a real application, use a database or cache)
task_status: Dict[str, Dict[str, str]] = {}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        return super(NumpyEncoder, self).default(obj)

class YouTubeLink(BaseModel):
    link: str


@app.post("/upload_video/")
async def upload_video(video: UploadFile = File(...)):
    video_bytes = await video.read()
    result = process_video(video_bytes, checkpoint,  resolution_width, resolution_height, pp_radius, pp_maxdists, num_points_lines, coordinates_data)
    json_compatible_result = json.loads(json.dumps(result, cls=NumpyEncoder))
    return JSONResponse(content=json_compatible_result)


@app.post("/submit_link")
async def receive_link(link: YouTubeLink, background_tasks: BackgroundTasks):
    youtube_link = link.link
    video_bytes = youtube_download(youtube_link).getvalue()

    # Generate a unique task ID
    task_id = str(uuid4())
    task_status[task_id] = {"status": "processing", "result": None}

    # Background task to process video
    background_tasks.add_task(
        process_video_with_status_update,
        task_id,
        video_bytes,
        checkpoint,
        resolution_width,
        resolution_height,
        pp_radius,
        pp_maxdists,
        num_points_lines,
        coordinates_data
    )

    # Return a response indicating the task has started
    return JSONResponse(content={"task_id": task_id, "status": "processing"}, status_code=202)


@app.get("/task_status/{task_id}")
async def get_task_status(task_id: str):
    task_info = task_status.get(task_id, {"status": "not found", "result": None})
    json_compatible_result = json.loads(json.dumps(task_info["result"], cls=NumpyEncoder))
    return JSONResponse(content={"task_id": task_id, "status": task_info["status"], "result": json_compatible_result})

async def main():
    ngrok.set_auth_token("2hUferPyUPiVQPdkthGYz7x4NTG_e2HiUntTDSKcRpYPcsDc")
    ngrok_tunnel = ngrok.connect(8000)
    print(f"FastAPI URL: {ngrok_tunnel.public_url}")

    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    asyncio.run(main())


FastAPI URL: https://255a-34-91-13-152.ngrok-free.app


INFO:     Started server process [9850]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[youtube] Extracting URL: https://youtu.be/3AVLwct6V0E?si=VMk3nsgLII8y2kHq
[youtube] 3AVLwct6V0E: Downloading webpage
[youtube] 3AVLwct6V0E: Downloading ios player API JSON
[youtube] 3AVLwct6V0E: Downloading web creator player API JSON
[youtube] 3AVLwct6V0E: Downloading m3u8 information
[info] 3AVLwct6V0E: Downloading 1 format(s): 18
[download] Destination: video_2kHq.mp4
[download] 100% of    1.56MiB in 00:00:00 at 11.11MiB/s  
INFO:     220.94.135.208:0 - "POST /submit_link HTTP/1.1" 202 Accepted
Loading model./train_59.pt
INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK
using cuda


  2%|██▉                                                                                                                       | 18/750 [00:07<04:43,  2.58it/s]

INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK


...

INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK


 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████   | 732/750 [05:23<00:10,  1.78it/s]

INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 750/750 [05:31<00:00,  2.26it/s]


INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK
INFO:     220.94.135.208:0 - "GET /task_status/bae4abd9-1477-4492-850a-7aa066e645d8 HTTP/1.1" 200 OK
